# Task 0: The Library of Babel
## Building a Dataset with Three Distinct Classes

**Goal**: Create a dataset where authorship (human vs AI) is the primary variable, NOT topic.

### Classes:
1. **Class 1**: Human-written text (from Project Gutenberg authors)
2. **Class 2**: AI-generated text on same topics (Gemini, neutral style)
3. **Class 3**: AI-generated text mimicking the author's style

In [1]:
!pip install requests pandas numpy google-generativeai nltk spacy textstat scikit-learn -q
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 64.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create a project folder
PROJECT_PATH = '/content/drive/MyDrive/Precog_NLP_Task_FIXED'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import requests
import pandas as pd
import numpy as np
import re
import time
import google.generativeai as genai
import spacy
from collections import Counter
import warnings

warnings.filterwarnings('ignore')

nlp = spacy.load('en_core_web_sm')
np.random.seed(42)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## Step 1: Configure Gemini API

In [4]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel('gemma-3-27b-it')

print("✓ Gemini API configured successfully")

✓ Gemini API configured successfully


## Step 2: Fetch and Clean Novels from Project Gutenberg

In [5]:
def get_clean_text(url):
    """
    Fetch text from Project Gutenberg and remove boilerplate.
    """
    response = requests.get(url)
    response.raise_for_status()
    text = response.text

    # Normalize characters
    text = re.sub(r'’', '\'', text, flags=re.DOTALL)
    text = re.sub(r'_', '', text, flags=re.DOTALL)
    text = re.sub(r'—', ' ', text, flags=re.DOTALL)
    text = re.sub(r'“', '\"', text, flags=re.DOTALL)
    text = re.sub(r'\[_Copyright.*?\]', '', text, flags=re.DOTALL)
    text = re.sub(r'\[Illustration.*?\]', '', text, flags=re.DOTALL)
    text = re.sub(r',', ',', text, flags=re.DOTALL)
    text = re.sub(r'!', '!', text, flags=re.DOTALL)
    text = re.sub(r'\'', "'", text)
    text = re.sub(r'["""]', '"', text)
    text = re.sub(r'—', ' -- ', text)
    text = re.sub(r'_', '', text)

    # Remove Gutenberg headers/footers
    start_markers = [
        r'\*\*\* START OF THIS PROJECT GUTENBERG',
        r'\*\*\*START OF THE PROJECT GUTENBERG',
    ]
    end_markers = [
        r'\*\*\* END OF THIS PROJECT GUTENBERG',
        r'\*\*\*END OF THE PROJECT GUTENBERG',
    ]

    for marker in start_markers:
        if re.search(marker, text):
            text = re.split(marker, text, maxsplit=1)[1]
            break

    for marker in end_markers:
        if re.search(marker, text):
            text = re.split(marker, text, maxsplit=1)[0]
            break

    # Remove chapter markers
    text = re.sub(r'CHAPTER [IVXLCDM]+\.?\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'Chapter \d+\.?\s*', '', text, flags=re.IGNORECASE)

    # Remove illustration markers
    text = re.sub(r'\[Illustration.*?\]', '', text, flags=re.DOTALL)

    # Clean whitespace
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()

    return text

# URLs for novels
AUSTEN_URLS = {
    'Pride and Prejudice': 'https://www.gutenberg.org/files/1342/1342-0.txt',
    'Sense and Sensibility': 'https://www.gutenberg.org/files/161/161-0.txt',
    'Emma': 'https://www.gutenberg.org/files/158/158-0.txt'
}

DICKENS_URLS = {
    'Great Expectations': 'https://www.gutenberg.org/files/1400/1400-0.txt',
    'Oliver Twist': 'https://www.gutenberg.org/files/730/730-0.txt',
    'A Tale of Two Cities': 'https://www.gutenberg.org/files/98/98-0.txt'
}

print("Fetching novels from Project Gutenberg...")

austen_texts = {}
for title, url in AUSTEN_URLS.items():
    print(f"  Downloading: {title}")
    austen_texts[title] = get_clean_text(url)
    time.sleep(1)

dickens_texts = {}
for title, url in DICKENS_URLS.items():
    print(f"  Downloading: {title}")
    dickens_texts[title] = get_clean_text(url)
    time.sleep(1)

# Combine all text per author
austen_full = ' '.join(austen_texts.values())
dickens_full = ' '.join(dickens_texts.values())

print(f"\n✓ Austen: {len(austen_full):,} characters")
print(f"✓ Dickens: {len(dickens_full):,} characters")

Fetching novels from Project Gutenberg...
  Downloading: Pride and Prejudice
  Downloading: Sense and Sensibility
  Downloading: Emma
  Downloading: Great Expectations
  Downloading: Oliver Twist
  Downloading: A Tale of Two Cities

✓ Austen: 2,271,477 characters
✓ Dickens: 2,647,096 characters


## Step 4: Extract Thematic Exposition from Novels \

In [6]:
def extract_thematic_exposition(text, author_name, min_words=100, max_words=200):
    """
    Extracts high-quality thematic exposition using spaCy.
    Filters out dialogue and ensures non-overlapping, contiguous chunks.
    """
    # Increase max_length for full-book processing
    nlp.max_length = len(text) + 100

    # Thematic keywords indicating authorial exposition
    thematic_keywords = {
        'society', 'class', 'marriage', 'fortune', 'family', 'duty',
        'rank', 'station', 'gentleman', 'lady', 'connexion', 'establishment',
        'situation', 'circumstance', 'consequence', 'propriety', 'distinction',
        'character', 'disposition', 'inclination', 'attachment', 'connection',
        'must', 'ought', 'should', 'naturally', 'generally', 'universally'
    }

    # Process the entire text once to avoid cutting sentences at chunk boundaries
    doc = nlp(text)

    all_passages = []
    current_passage = []
    current_word_count = 0

    for sent in doc.sents:
        sent_text = sent.text.strip()

        # 1. Refined Dialogue Check: Look for actual quotes, not just any apostrophe
        # This prevents "don't" or "it's" from being flagged as dialogue
        if re.search(r'["""]', sent_text):
            # Reset if we hit dialogue to ensure passages are contiguous exposition
            current_passage = []
            current_word_count = 0
            continue

        # 2. Word counting (excluding whitespace/punctuation)
        words = [t.text for t in sent if not t.is_space and not t.is_punct]
        word_count = len(words)

        # 3. Skip fragments
        if word_count < 5:
            continue

        # 4. Building the passage
        if current_word_count + word_count <= max_words:
            # Check if at least one sentence in the building passage is thematic
            sent_lower = sent_text.lower()
            is_thematic = any(k in sent_lower for k in thematic_keywords)

            if is_thematic or current_word_count > 0:
                current_passage.append(sent_text)
                current_word_count += word_count

        # 5. Finalizing the passage
        if current_word_count >= min_words:
            all_passages.append(' '.join(current_passage))
            current_passage = []
            current_word_count = 0

    df = pd.DataFrame({
        'text': all_passages,
        'author': author_name,
        'class': 'human'
    })

    # Final word count check
    df['word_count'] = df['text'].apply(lambda x: len(x.split()))
    return df[(df['word_count'] >= min_words) & (df['word_count'] <= max_words)].reset_index(drop=True)

# --- CONSOLIDATED EXECUTION AND 500-SAMPLE GUARANTEE ---

print("Processing Austen...")
austen_exposition = extract_thematic_exposition(austen_full, 'Austen')

print("Processing Dickens...")
dickens_exposition = extract_thematic_exposition(dickens_full, 'Dickens')

# Combine and Shuffle
class1_df = pd.concat([austen_exposition, dickens_exposition], ignore_index=True)
class1_df = class1_df.sample(frac=1, random_state=42).reset_index(drop=True)

# The 2000-Sample Guarantee
TARGET = 2000
if len(class1_df) >= TARGET:
    class1_df = class1_df.head(TARGET)
    print(f"\n✓ Successfully extracted and truncated to {TARGET} samples.")
else:
    print(f"\n Warning: Only found {len(class1_df)} samples. Add more text sources.")

# Show final counts
print(class1_df['author'].value_counts())

Processing Austen...
Processing Dickens...

author
Austen     1160
Dickens     766
Name: count, dtype: int64


## Step 3: Extract Topics from Human Text

In [7]:
def extract_topics_with_gemini(text_sample, num_topics, start, end):
    """
    Uses Gemini to extract core topics from the text.
    """
    prompt = f"""
    <start_of_turn>user
    You are an expert Literary Scholar specializing in thematic decomposition.

    TASK:
    Analyze the provided literary excerpt and extract exactly {num_topics} core substantive themes. It is imperative that these themes must be central to the text.

    CONSTRAINTS:
    - Focus exclusively on abstract, higher-order themes (e.g., "social hierarchy," "unrequited affection," "industrial alienation").
    - DO NOT list plot points, character names, or specific events.
    - Each theme must be a concise 2-3 word label.
    - Output ONLY a numbered list. No introductory or concluding remarks.

    TEXT EXCERPT:
    ---
    {text_sample[start:end]}
    ---

    OUTPUT FORMAT:
    1. [Theme Name]
    2. [Theme Name]
    ...
    <end_of_turn>
    <start_of_turn>model
    """

    try:
        response = model.generate_content(prompt)
        topics_text = response.text

        # Parse the numbered list
        topics = []
        for line in topics_text.split('\n'):
            # Match lines like "1. Topic" or "1) Topic"
            match = re.match(r'^\d+[.)\s]+(.*?)$', line.strip())
            if match:
                topic = match.group(1).strip()
                if topic:
                    topics.append(topic)

        return topics[:num_topics]

    except Exception as e:
        print(f"Error extracting topics: {e}")
        return []


# Extract topics from both authors
print("Extracting topics using Gemini...\n")

# Sample text from each author for topic extraction
austen_sample = ' '.join(class1_df[class1_df['author'] == 'Austen']['text'].head(250))
dickens_sample = ' '.join(class1_df[class1_df['author'] == 'Dickens']['text'].head(250))

topics_austen, topics_dickens = [], []

for i in range(2):
  topicsAusten = extract_topics_with_gemini(austen_sample, num_topics=2+(i%2), start=i*5000, end=i*10000+4000)
  time.sleep(2)  # Rate limiting
  topicsDickens = extract_topics_with_gemini(dickens_sample, num_topics=2+(i%2), start=i*5000, end=i*10000+4000)
  time.sleep(2)
  topics_austen.extend(topicsAusten), topics_dickens.extend(topicsDickens)

all_topics = topics_austen + topics_dickens

print(f"\n✓ Extracted {len(all_topics)} topics:")
for i, topic in enumerate(all_topics, 1):
    print(f"  {i}. {topic}")

# Save topics for later use
topics_list = all_topics

Extracting topics using Gemini...


✓ Extracted 10 topics:
  1. Social Expectations
  2. Self-Deception
  3. Social Class
  4. Romantic Disillusionment
  5. Familial Duty
  6. Social Inequality
  7. Moral Corruption
  8. Social Stratification
  9. Moral Corruption
  10. Lost Innocence


## Step 5: Tag Human Passages with Topics

Now we need to determine which topic each human passage discusses.

In [8]:
def assign_topic_to_passage(passage, topics_list):
    """
    Use LLM to determine which topic a passage primarily discusses.
    """
    prompt = f"""
Given this passage from a Victorian novel:

\"{passage}\"

Which ONE of these topics does it primarily discuss?

Topics:
{chr(10).join(f'{i+1}. {topic}' for i, topic in enumerate(topics_list))}

Respond with ONLY the topic name, nothing else.
"""

    try:
        response = model.generate_content(prompt)
        assigned_topic = response.text.strip()

        # Match to actual topic (fuzzy)
        for topic in topics_list:
            if topic.lower() in assigned_topic.lower() or assigned_topic.lower() in topic.lower():
                return topic

        # If no match, return first topic as fallback
        return topics_list[0]
    except Exception as e:
        print(f"Error assigning topic: {e}")
        return topics_list[0]

# Sample passages to tag (we'll do a subset to save API calls)
# Then randomly assign topics to others based on distribution

print("Assigning topics to human passages (sampling for efficiency)...")

# Tag 50 passages from each author with LLM
sample_size = min(50, len(austen_exposition), len(dickens_exposition))

austen_sample_tagged = []
for i in range(sample_size):
    passage = austen_exposition.iloc[i]['text']
    topic = assign_topic_to_passage(passage, all_topics)
    austen_sample_tagged.append(topic)
    time.sleep(5)  # Rate limiting
    if (i+1) % 10 == 0:
        print(f"  Austen: {i+1}/{sample_size}")

dickens_sample_tagged = []
for i in range(sample_size):
    passage = dickens_exposition.iloc[i]['text']
    topic = assign_topic_to_passage(passage, all_topics)
    dickens_sample_tagged.append(topic)
    time.sleep(5)
    if (i+1) % 10 == 0:
        print(f"  Dickens: {i+1}/{sample_size}")

# Get topic distribution from samples
topic_distribution = Counter(austen_sample_tagged + dickens_sample_tagged)
print(f"\nTopic distribution from tagged samples:")
for topic, count in topic_distribution.most_common():
    print(f"  {topic}: {count}")

# Assign topics to all passages based on this distribution
def assign_topics_by_distribution(df, topics_list, distribution):
    # Get counts for all topics, using 0 for those not in the distribution
    topic_counts = [distribution.get(t, 0) for t in topics_list]

    # Calculate the sum of these counts
    total_counts = sum(topic_counts)

    if total_counts == 0:
        # If no topics were found in the sample, assign equal probability
        probs = [1 / len(topics_list)] * len(topics_list)
    else:
        # Convert counts to probabilities
        probs = [count / total_counts for count in topic_counts]

    # Assign topics randomly based on distribution
    assigned = np.random.choice(topics_list, size=len(df), p=probs)
    return assigned

austen_exposition['topic'] = assign_topics_by_distribution(austen_exposition, all_topics, topic_distribution)
dickens_exposition['topic'] = assign_topics_by_distribution(dickens_exposition, all_topics, topic_distribution)

print("\n✓ Topics assigned to all passages")

Assigning topics to human passages (sampling for efficiency)...
  Austen: 10/50
  Austen: 20/50
  Austen: 30/50
  Austen: 40/50
  Austen: 50/50
  Dickens: 10/50
  Dickens: 20/50
  Dickens: 30/50
  Dickens: 40/50
  Dickens: 50/50

Topic distribution from tagged samples:
  Social Expectations: 27
  Social Class: 24
  Social Inequality: 10
  Lost Innocence: 9
  Moral Corruption: 8
  Social Stratification: 6
  Familial Duty: 6
  Romantic Disillusionment: 5
  Self-Deception: 5

✓ Topics assigned to all passages


## Step 4: Generate AI Text - Class 2 (Neutral Style)

In [10]:
from prompt import *

In [11]:
# Generate Class 2: AI text in neutral style
print("\n" + "="*60)
print("GENERATING CLASS 2: AI TEXT (NEUTRAL STYLE)")
print("="*60)

class2_df = generate_ai_dataset_optimized(
    model = model,
    topics=topics_list,
    num_samples_per_topic=50,  # 50 samples per topic
    style="neutral",
    batch_size=5
)

print(f"\n✓ Class 2 (AI Neutral) Dataset Created:")
print(f"  - Total samples: {len(class2_df)}")
print(f"  - Avg words per sample: {class2_df['word_count'].mean():.1f}")
print(f"  - Word count range: {class2_df['word_count'].min()}-{class2_df['word_count'].max()}")

check_dataset_diversity(class2_df)


GENERATING CLASS 2: AI TEXT (NEUTRAL STYLE)
VARIETY MODE ENABLED
 - Using varied batch generation to minimize duplicates
 - Explicit uniqueness instructions per batch

Generating 500 samples (50 per topic)...
Using batch generation: 5 paragraphs per API call
Total API calls needed: 100 (vs 500 without batching)
Time savings: ~10.0 minutes


TOPIC 1/10: Social Expectations
  Generating 50 VARIED samples...
  API calls needed: 10 (batch size: 5)
  API Call 1/10: Generating 5 variations... ✓ (5/50 total)
  API Call 2/10: Generating 5 variations... ✓ (10/50 total)
  API Call 3/10: Generating 5 variations... ✓ (15/50 total)
  API Call 4/10: Generating 5 variations... ✓ (20/50 total)
  API Call 5/10: Generating 5 variations... ✓ (25/50 total)
  API Call 6/10: Generating 5 variations... ✓ (30/50 total)
  API Call 7/10: Generating 5 variations... ✓ (35/50 total)
  API Call 8/10: Generating 5 variations... ✓ (40/50 total)
  API Call 9/10: Generating 5 variations... ✓ (45/50 total)
  API Call 1

ERROR:tornado.access:503 POST /v1beta/models/gemma-3-27b-it:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 29728.04ms


 ✓ (40/50 total)
  API Call 9/10: Generating 5 variations... ✓ (45/50 total)
  API Call 10/10: Generating 5 variations... ✓ (50/50 total)

TOPIC 3/10: Social Class
  Generating 50 VARIED samples...
  API calls needed: 10 (batch size: 5)
  API Call 1/10: Generating 5 variations... ✓ (5/50 total)
  API Call 2/10: Generating 5 variations... ✓ (10/50 total)
  API Call 3/10: Generating 5 variations... ✓ (15/50 total)
  API Call 4/10: Generating 5 variations... ✓ (20/50 total)
  API Call 5/10: Generating 5 variations... ✓ (25/50 total)
  API Call 6/10: Generating 5 variations... ✓ (30/50 total)
  API Call 7/10: Generating 5 variations... ✓ (35/50 total)
  API Call 8/10: Generating 5 variations... ✓ (40/50 total)
  API Call 9/10: Generating 5 variations... ✓ (45/50 total)
  API Call 10/10: Generating 5 variations... ✓ (50/50 total)

TOPIC 4/10: Romantic Disillusionment
  Generating 50 VARIED samples...
  API calls needed: 10 (batch size: 5)
  API Call 1/10: Generating 5 variations... ✓ (5/50 

In [12]:
# Preview Class 2 samples
print("\n=== Sample AI Text (Neutral Style) ===")
print(class2_df.iloc[490]['text'])


=== Sample AI Text (Neutral Style) ===
Is it a universal truth that growing up necessitates a severing from naive belief? The erosion of innocence isn’t a single, dramatic event, but a gradual accumulation of disillusionment. It’s the slow realization that adults aren’t always benevolent protectors, that systems aren’t inherently just, and that the world operates on complexities far removed from the simple narratives of childhood. This isn’t necessarily a negative process; it’s a necessary one for developing critical thinking and empathy. However, the pain of that initial fracture – the moment a cherished ideal shatters – can be profoundly unsettling. It forces a confrontation with the ambiguities of existence, demanding a more nuanced, and often more cynical, worldview. The challenge lies in retaining compassion *despite* understanding the darkness.


## Step 5: Generate AI Text - Class 3 (Mimicked Style)

In [13]:
# Generate Class 3: AI text mimicking Austen's style
print("\n" + "="*60)
print("GENERATING CLASS 3A: AI TEXT (MIMICKING AUSTEN)")
print("="*60)

class3a_df = generate_ai_dataset_optimized(
    model,
    topics=topics_austen,
    num_samples_per_topic=50,  # 50 samples per topic
    style="mimicked",
    author_name="Austen",
    batch_size=5
)

print(f"\n✓ Class 3A (AI Mimicking Austen) Dataset Created:")
print(f"  - Total samples: {len(class3a_df)}")
print(f"  - Avg words per sample: {class3a_df['word_count'].mean():.1f}")

check_dataset_diversity(class3a_df)


GENERATING CLASS 3A: AI TEXT (MIMICKING AUSTEN)
VARIETY MODE ENABLED
 - Using varied batch generation to minimize duplicates
 - Explicit uniqueness instructions per batch

Generating 250 samples (50 per topic)...
Using batch generation: 5 paragraphs per API call
Total API calls needed: 50 (vs 250 without batching)
Time savings: ~5.0 minutes


TOPIC 1/5: Social Expectations
  Generating 50 VARIED samples...
  API calls needed: 10 (batch size: 5)
  API Call 1/10: Generating 5 variations... ✓ (5/50 total)
  API Call 2/10: Generating 5 variations... ✓ (10/50 total)
  API Call 3/10: Generating 5 variations... ✓ (15/50 total)
  API Call 4/10: Generating 5 variations... ✓ (20/50 total)
  API Call 5/10: Generating 5 variations... ✓ (25/50 total)
  API Call 6/10: Generating 5 variations... ✓ (30/50 total)
  API Call 7/10: Generating 5 variations... ✓ (35/50 total)
  API Call 8/10: Generating 5 variations... ✓ (40/50 total)
  API Call 9/10: Generating 5 variations... ✓ (45/50 total)
  API Call 

In [14]:
# Generate Class 3: AI text mimicking Dickens's style
print("\n" + "="*60)
print("GENERATING CLASS 3B: AI TEXT (MIMICKING DICKENS)")
print("="*60)

class3b_df = generate_ai_dataset_optimized(
    model,
    topics=topics_dickens,
    num_samples_per_topic=50,  # 50 samples per topic
    style="mimicked",
    author_name="Dickens",
    batch_size=5
)

print(f"\n✓ Class 3B (AI Mimicking Dickens) Dataset Created:")
print(f"  - Total samples: {len(class3b_df)}")
print(f"  - Avg words per sample: {class3b_df['word_count'].mean():.1f}")

check_dataset_diversity(class3b_df)


GENERATING CLASS 3B: AI TEXT (MIMICKING DICKENS)
VARIETY MODE ENABLED
 - Using varied batch generation to minimize duplicates
 - Explicit uniqueness instructions per batch

Generating 250 samples (50 per topic)...
Using batch generation: 5 paragraphs per API call
Total API calls needed: 50 (vs 250 without batching)
Time savings: ~5.0 minutes


TOPIC 1/5: Social Inequality
  Generating 50 VARIED samples...
  API calls needed: 10 (batch size: 5)
  API Call 1/10: Generating 5 variations... ✓ (5/50 total)
  API Call 2/10: Generating 5 variations... ✓ (10/50 total)
  API Call 3/10: Generating 5 variations... ✓ (15/50 total)
  API Call 4/10: Generating 5 variations... ✓ (20/50 total)
  API Call 5/10: Generating 5 variations... ✓ (25/50 total)
  API Call 6/10: Generating 5 variations... ✓ (30/50 total)
  API Call 7/10: Generating 5 variations... ✓ (35/50 total)
  API Call 8/10: Generating 5 variations... ✓ (40/50 total)
  API Call 9/10: Generating 5 variations... ✓ (45/50 total)
  API Call 1

In [15]:
# Combine Class 3 datasets
class3_df = pd.concat([class3a_df, class3b_df], ignore_index=True)

print(f"\n✓ Class 3 (AI Mimicked) Complete Dataset:")
print(f"  - Total samples: {len(class3_df)}")
print(f"  - Austen-style samples: {len(class3a_df)}")
print(f"  - Dickens-style samples: {len(class3b_df)}")


✓ Class 3 (AI Mimicked) Complete Dataset:
  - Total samples: 500
  - Austen-style samples: 250
  - Dickens-style samples: 250


In [16]:
# Preview Class 3 samples
print("\n=== Sample AI Text (Mimicking Austen) ===")
print(class3a_df.iloc[239]['text'])

print("\n=== Sample AI Text (Mimicking Dickens) ===")
print(class3b_df.iloc[239]['text'])


=== Sample AI Text (Mimicking Austen) ===
The notion of familial duty, when divorced from affection and tempered by reason, can become a most oppressive force. Consider the plight of Miss Dorothea Grey, whose uncle, a man of considerable wealth and even more considerable eccentricity, had decreed that she must reside with him for the remainder of his days, simply to provide him with the comfort of female companionship. It was not a request, but a command, backed by the threat of disinheritance. Miss Grey, a woman of independent spirit and intellectual pursuits, found herself confined to a life of monotonous routine, her talents wasted, her aspirations stifled. The irony, of course, was that her uncle, in attempting to secure his own happiness, had rendered hers utterly miserable, demonstrating the perilous consequences of elevating obligation above all other considerations.

=== Sample AI Text (Mimicking Dickens) ===
One might be tempted to locate the loss of innocence in a singular, 

## Step 6: Combine All Classes and Create Final Dataset

In [17]:
# Standardize columns across all dataframes
def standardize_dataframe(df, source_class):
    """
    Ensures all dataframes have the same columns.
    """
    standard_df = pd.DataFrame()
    standard_df['text'] = df['text']
    standard_df['word_count'] = df['word_count']
    standard_df['class'] = source_class
    standard_df['author'] = df['author']

    # Add topic if it exists
    if 'topic' in df.columns:
        standard_df['topic'] = df['topic']
    else:
        standard_df['topic'] = 'original_literature'

    return standard_df

# Standardize all datasets
class1_standard = standardize_dataframe(class1_df, 'human')
class2_standard = standardize_dataframe(class2_df, 'ai_neutral')
class3_standard = standardize_dataframe(class3_df, 'ai_mimicked')

# Combine all three classes
final_dataset = pd.concat([
    class1_standard,
    class2_standard,
    class3_standard
], ignore_index=True)

# Shuffle the dataset
final_dataset = final_dataset.sample(frac=1, random_state=42).reset_index(drop=True)

# Add a unique ID to each sample
final_dataset['sample_id'] = range(len(final_dataset))

print("="*60)
print("FINAL DATASET SUMMARY")
print("="*60)
print(f"\nTotal samples: {len(final_dataset)}")
print(f"\nClass distribution:")
print(final_dataset['class'].value_counts())
print(f"\nAuthor distribution:")
print(final_dataset['author'].value_counts())
print(f"\nWord count statistics:")
print(final_dataset.groupby('class')['word_count'].describe())

FINAL DATASET SUMMARY

Total samples: 2926

Class distribution:
class
human          1926
ai_mimicked     500
ai_neutral      500
Name: count, dtype: int64

Author distribution:
author
Austen              1160
Dickens              766
AI                   500
Austen_mimicked      250
Dickens_mimicked     250
Name: count, dtype: int64

Word count statistics:
              count        mean        std    min    25%    50%    75%    max
class                                                                        
ai_mimicked   500.0  132.454000  12.151345  101.0  124.0  131.0  141.0  168.0
ai_neutral    500.0  111.040000  10.038323   80.0  104.0  110.0  118.0  142.0
human        1926.0  119.667705  17.041970  100.0  106.0  115.0  128.0  198.0


## Step 7: Save the Dataset

In [18]:

# Also save individual class datasets for reference
class1_standard.to_csv(os.path.join(PROJECT_PATH, 'class1_human.csv'), index=False)
class2_standard.to_csv(os.path.join(PROJECT_PATH,'class2_ai_neutral.csv'), index=False)
class3_standard.to_csv(os.path.join(PROJECT_PATH,'class3_ai_mimicked.csv'), index=False)
print("✓ Individual class datasets saved")

# Save topics list
with open(os.path.join(PROJECT_PATH,'extracted_topics.txt'), 'w') as f:
    for i, topic in enumerate(topics_list, 1):
        f.write(f"{i}. {topic}\n")
print("✓ Topics list saved to 'extracted_topics.txt'")

✓ Individual class datasets saved
✓ Topics list saved to 'extracted_topics_alt.txt'


## Step 8: Basic Dataset Validation

In [19]:
# 1. Load the CSV Classes into DataFrames
class1_df = pd.read_csv(os.path.join(PROJECT_PATH, 'class1_human.csv'))
class2_df = pd.read_csv(os.path.join(PROJECT_PATH, 'class2_ai_neutral.csv'))
class3_df = pd.read_csv(os.path.join(PROJECT_PATH, 'class3_ai_mimicked.csv'))

# Combine all three classes
final_dataset = pd.concat([
    class1_df,
    class2_df,
    class3_df
], ignore_index=True)

# Shuffle the dataset
final_dataset = final_dataset.sample(frac=1, random_state=42).reset_index(drop=True)

# Add a unique ID to each sample
final_dataset['sample_id'] = range(len(final_dataset))

final_dataset.to_csv(os.path.join(PROJECT_PATH,'complete_dataset.csv'), index=False)

finalD = pd.read_csv(os.path.join(PROJECT_PATH,'complete_dataset.csv'))
# 2. Load the Extracted Topics text file
with open(os.path.join(PROJECT_PATH, 'extracted_topics.txt'), 'r') as f:
    topics_list = [line.strip() for line in f.readlines() if line.strip()]

# Verification
print(f"Successfully loaded Class 1: {len(class1_df)} samples")
print(f"Successfully loaded Class 2: {len(class2_df)} samples")
print(f"Successfully loaded Class 3: {len(class3_df)} samples")
print(f"Successfully loaded {len(topics_list)} topics.")

Successfully loaded Class 1: 1926 samples
Successfully loaded Class 2: 500 samples
Successfully loaded Class 3: 500 samples
Successfully loaded 10 topics.


In [20]:
# Check for any anomalies
print("\n" + "="*60)
print("DATASET VALIDATION")
print("="*60)

# Check for missing values
print("\nMissing values:")
print(final_dataset.isnull().sum())

# Check text length distribution
print("\nText length validation:")
too_short = final_dataset[final_dataset['word_count'] < 100]
too_long = final_dataset[final_dataset['word_count'] > 200]
print(f"  - Samples < 100 words: {len(too_short)}")
print(f"  - Samples > 200 words: {len(too_long)}")

# Check for duplicates
duplicates = final_dataset[final_dataset.duplicated(subset=['text'], keep=False)]
print(f"\nDuplicate texts: {len(duplicates)}")

print("\n✓ Task 0 Complete!")


DATASET VALIDATION

Missing values:
text          0
word_count    0
class         0
author        0
topic         0
sample_id     0
dtype: int64

Text length validation:
  - Samples < 100 words: 46
  - Samples > 200 words: 0

Duplicate texts: 70

✓ Task 0 Complete!


In [21]:
# Display a few random samples from each class for visual inspection
print("\n" + "="*60)
print("SAMPLE INSPECTION")
print("="*60)

for class_name in ['human', 'ai_neutral', 'ai_mimicked']:
    print(f"\n{'='*60}")
    print(f"CLASS: {class_name.upper()}")
    print(f"{'='*60}")
    samples = final_dataset[final_dataset['class'] == class_name].sample(n=2, random_state=42)

    for idx, row in samples.iterrows():
        print(f"\nAuthor: {row['author']} | Words: {row['word_count']}")
        print(f"Topic: {row['topic']}")
        print(f"Text: {row['text'][:400]}...")
        print("-" * 60)


SAMPLE INSPECTION

CLASS: HUMAN

Author: Dickens | Words: 121
Topic: original_literature
Text: He was a broadshouldered loose-limbed swarthy fellow of great strength, never in a hurry, and always slouching. He never even seemed to come to his work on purpose, but would slouch in as if by mere accident; and when he went to the Jolly Bargemen to eat his dinner, or went away at night, he would slouch out, like Cain or the Wandering Jew, as if he had no idea where he was going and no intention ...
------------------------------------------------------------

Author: Austen | Words: 103
Topic: original_literature
Text: As long as Mr. Knightley remained with them, Emma's fever continued; but when he was gone, she began to be a little tranquillised and subdued and in the course of the sleepless night, which was the tax for such an evening, she found one or two such very serious points to consider, as made her feel, that even her happiness must have some alloy. She could not be alone without 